<a href="https://colab.research.google.com/github/SanduniHerath/diabetic-retinopathy-cv-assignment/blob/feature%2Fphase-5-training-evaluation/notebooks/diabetic_retinopathy_training_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone -b feature/phase-5-training-evaluation https://github.com/SanduniHerath/diabetic-retinopathy-cv-assignment.git
%cd diabetic-retinopathy-cv-assignment

Cloning into 'diabetic-retinopathy-cv-assignment'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 144 (delta 39), reused 118 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 28.96 MiB | 23.92 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/diabetic-retinopathy-cv-assignment


In [2]:
!pip install torch torchvision numpy matplotlib opencv-python scikit-learn

In [3]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"sandunisathsarani","key":"0c6b2bce0e7c01e5221ee0d46e678f7f"}'}

In [4]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
!pip install kaggle

In [6]:
!kaggle competitions download -c aptos2019-blindness-detection -p data/raw

100% 9.51G/9.51G [01:39<00:00, 103MB/s]



In [7]:
!unzip -q data/raw/aptos2019-blindness-detection.zip -d data/raw

In [8]:
!rm -rf data/raw/test_images

In [9]:
!cp data/raw/train.csv data/raw/train_images/train.csv

In [10]:
!python src/organize_dataset.py


[1] Loading CSV from: /content/diabetic-retinopathy-cv-assignment/data/raw/train_images/train.csv
     Loaded 3662 records | Columns: ['id_code', 'diagnosis']

[2] Creating class directories under: /content/diabetic-retinopathy-cv-assignment/data/organized
     No_DR                -> /content/diabetic-retinopathy-cv-assignment/data/organized/No_DR
     Mild                 -> /content/diabetic-retinopathy-cv-assignment/data/organized/Mild
     Moderate             -> /content/diabetic-retinopathy-cv-assignment/data/organized/Moderate
     Severe               -> /content/diabetic-retinopathy-cv-assignment/data/organized/Severe
     Proliferative_DR     -> /content/diabetic-retinopathy-cv-assignment/data/organized/Proliferative_DR

[3] Organising 3662 images into class folders ...
  Organised 500/3662 images ...
  Organised 1000/3662 images ...
  Organised 1500/3662 images ...
  Organised 2000/3662 images ...
  Organised 2500/3662 images ...
  Organised 3000/3662 images ...
  Organise

In [11]:
!python src/split_dataset.py

PHASE 1: STRATIFIED TRAIN / VAL / TEST DATASET SPLIT
  Input Directory   : data/organized
  Output Directory  : data/split
  Ratios (T/V/Te)   : 0.70 / 0.15 / 0.15
  Random Seed       : 42
  Dry Run           : False
--------------------------------------------------------------------------------
[INFO] Discovered 3662 total images across 5 classes.

           APTOS 2019 DATASET STRATIFIED TRAIN / VAL / TEST SPLIT SUMMARY
Label  | Class Name        | Total   | Train (70%)    | Val (15%)      | Test (15%)    
-------------------------------------------------------------------------------------
0      | No_DR             | 1805    |  1264 ( 70.0%) |   271 ( 15.0%) |   270 ( 15.0%)
1      | Mild              | 370     |   259 ( 70.0%) |    56 ( 15.1%) |    55 ( 14.9%)
2      | Moderate          | 999     |   699 ( 70.0%) |   150 ( 15.0%) |   150 ( 15.0%)
3      | Severe            | 193     |   135 ( 69.9%) |    29 ( 15.0%) |    29 ( 15.0%)
4      | Proliferative_DR  | 295     |   206 ( 

In [12]:
!rm -rf data/organized

In [13]:
!rm -rf data/raw/train_images

In [14]:
!python src/augmentation.py


  TRAINING SET -- BEFORE AUGMENTATION (data/split/train/)
  Class                   Count   Percentage   Distribution Bar
  -------------------- --------   ----------   ------------------------
  No_DR                   1,264        49.3%   ########################
  Mild                      259        10.1%   ####
  Moderate                  699        27.3%   #############
  Severe                    135         5.3%   ##
  Proliferative_DR          206         8.0%   ###
  -------------------- --------   ----------   ------------------------
  TOTAL                   2,563       100.0%
  Imbalance Ratio (Max / Min): 9.36x


  TRAINING SET -- AFTER AUGMENTATION (data/split/train_augmented/)
  Class                   Count   Percentage   Distribution Bar
  -------------------- --------   ----------   ------------------------
  No_DR                   1,264        19.7%   #####################
  Mild                    1,295        20.2%   ######################
  Moderate           

In [15]:
!python src/train.py --data-root data/split --output-dir reports/training --stage1-epochs 3 --stage2-epochs 4 --batch-size 32 --num-workers 2


[Device] Using: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB

[Mode] Single training run ...
[Data] Train: 6,408 images across 5 classes
[Data] Val  : 550  images
[Data] Classes (train): ['Mild', 'Moderate', 'No_DR', 'Proliferative_DR', 'Severe']
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 145MB/s]
[Loss] Class weights: {'No_DR': '0.990', 'Mild': '0.917', 'Moderate': '1.014', 'Severe': '1.037', 'Proliferative_DR': '1.055'}
/content/diabetic-retinopathy-cv-assignment/src/train.py:557: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if (use_amp and torch.cuda.is_available()) else None  # type: ignore[attr-defined]

 STAGE 1: Feature Extraction (Backbone Frozen)
   LR=0.001 | Epochs=3 | EarlyStop patience=5
/content/diabetic-ret

In [16]:
!python src/evaluate.py \
  --checkpoint reports/training/best_model_20260926_053343.pth \
  --data-root data/split \
  --output-dir reports/evaluation \
  --backbone efficientnet_b0


[Device] Using: cuda
[Data] Test: 549 images across 5 classes
[OK] Checkpoint loaded: reports/training/best_model_20260926_053343.pth

[Eval] Test Loss     : 0.6395
[Eval] Test Accuracy : 74.32%

 CLASSIFICATION REPORT
  Test Loss           : 0.6395
  Test Accuracy       : 74.32%
  Weighted F1         : 0.7446
  Macro F1            : 0.5812
  Quadratic Wtd Kappa : 0.9990

  Per-class breakdown:
  Class                   Precision   Recall       F1   Support
  -----------------------------------------------------------
  No_DR                      0.5254   0.5636   0.5439        55
  Mild                       0.7255   0.4933   0.5873       150
  Moderate                   0.9336   0.9889   0.9604       270
  Severe                     0.4889   0.4889   0.4889        45
  Proliferative_DR           0.2456   0.4828   0.3256        29
[OK] Classification report saved to: reports/evaluation/classification_report.txt
[OK] Metrics JSON saved to: reports/evaluation/metrics_summary.json
[Plot

In [17]:
from google.colab import files
import shutil

# Copy the timestamped checkpoint to standard 'best_model.pth' for the app
!cp reports/training/best_model_20260926_053343.pth reports/training/best_model.pth

# Zip all training and evaluation artifacts
!zip -r completed_reports.zip reports/training reports/evaluation

# Download to your computer
files.download('completed_reports.zip')

  adding: reports/training/ (stored 0%)
  adding: reports/training/loss_curve_20260926_053343.png (deflated 10%)
  adding: reports/training/.gitkeep (stored 0%)
  adding: reports/training/best_model_20260926_053343.pth (deflated 8%)
  adding: reports/training/lr_curve_20260926_053343.png (deflated 17%)
  adding: reports/training/accuracy_curve_20260926_053343.png (deflated 11%)
  adding: reports/training/best_model.pth (deflated 8%)
  adding: reports/evaluation/ (stored 0%)
  adding: reports/evaluation/.gitkeep (stored 0%)
  adding: reports/evaluation/metrics_summary.json (deflated 64%)
  adding: reports/evaluation/classification_report.txt (deflated 64%)
  adding: reports/evaluation/confusion_matrix_norm.png (deflated 12%)
  adding: reports/evaluation/confusion_matrix_raw.png (deflated 16%)
  adding: reports/evaluation/gradcam_samples/ (stored 0%)
  adding: reports/evaluation/gradcam_samples/gradcam_Proliferative_DR.png (deflated 0%)
  adding: reports/evaluation/gradcam_samples/gradca

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>